In [ ]:
# Scenarios Setup
# -------------------------------

import pickle
from pathlib import Path

import numpy as np

from app.analysis.distributions import Constant
from app.analysis.psa.parameter_resolver import ParameterResolver
from app.analysis.psa.parameters import Parameter
from app.analysis.psa.sampler import PSASampler
from app.domain.enums import HealthStates, Regime
from app.domain.inputs import ModelInput
from app.domain.scenario import Scenario, ScenarioBundle
from app.domain.worker import worker_function
from app.notebook.parameter_sets import HemophiliaParamRepo
from app.notebook.scenario_helpers import (
    define_scenario_extension,
    get_tornado_ranges,
    insert_scenario,
    ltb_mode_for_scenario,
)
from app.persistence.context import ModelContext
from engine.chains import Chain
from utils.logging import setup_root_logger
from utils.path_utils import get_project_root

logger = setup_root_logger()
context = ModelContext.load()
seed = context.simulation.environment.seed
sample_size = context.simulation.psa.development

root = get_project_root()

cost_unit = context.costs.currencies[0].code
per_unit_cost = context.costs.costs[0].pricing.per_unit[cost_unit]


# Helper functions
def mean(value):
    return np.mean(value)


def low(value):
    return np.percentile(value, 2.5)


def high(value):
    return np.percentile(value, 97.5)


param_repo = HemophiliaParamRepo(root=root, cache_path=Path("app/cache/samples.pkl"), context=context)
with open(param_repo.root / param_repo.cache_path, "rb") as f:
    samples = pickle.load(f)

# on_demand base scenario
owsa_parameters = param_repo.load_owsa_parameters()
psa_parameters = param_repo.load_psa_parameters()
owsa_keys = param_repo.ows_params_keys

tornado_ranges = get_tornado_ranges(psa_parameters, owsa_keys, sample_size, seed)


scenarios = []


bayesian_scenario = [
    # OWSA _ EARLY
    Scenario(name="early on_demand bayesian", regime=Regime.ON_DEMAND, overrides={}),
    Scenario(
        name="early prophylaxis bayesian",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "bleeding_rate": Parameter(
                Constant(mean(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(mean(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
]

# ---------------------------
#           OWSA
# ---------------------------
# Define low _ high scenarios
annual_bleeding_rate_scenarios = [
    Scenario(
        name="early on_demand bayesian abr_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "bleeding_rate": Parameter(Constant(low(samples["on_demand"]["bayesian"]))),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian abr_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "bleeding_rate": Parameter(
                Constant(low(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian abr_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(Constant(low(samples["on_demand"]["bayesian"]))),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian abr_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(low(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    Scenario(
        name="early on_demand bayesian abr_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "bleeding_rate": Parameter(
                Constant(high(samples["on_demand"]["bayesian"]))
            ),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian abr_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "bleeding_rate": Parameter(
                Constant(high(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian abr_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(high(samples["on_demand"]["bayesian"]))
            ),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian abr_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "bleeding_rate": Parameter(
                Constant(high(samples["prophylaxis"]["bayesian"]))
            ),
        },
    ),
]

weight_scenarios = [
    Scenario(
        name="early on_demand bayesian weight_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian weight_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian weight_low",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian weight_low",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=0.9)),
        },
    ),
    Scenario(
        name="early on_demand bayesian weight_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
    Scenario(
        name="early prophylaxis bayesian weight_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(value=10 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
    # OWSA _ PROPHYLAXIS
    Scenario(
        name="lifetime on_demand bayesian weight_high",
        regime=Regime.ON_DEMAND,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
    Scenario(
        name="lifetime prophylaxis bayesian weight_high",
        regime=Regime.PROPHYLAXIS,
        overrides={
            "cycles": Parameter(distribution=Constant(98 * 52)),
            "weight_factor": Parameter(distribution=Constant(value=1.1)),
        },
    ),
]


scenarios.extend(bayesian_scenario)
scenarios.extend(weight_scenarios)
scenarios.extend(annual_bleeding_rate_scenarios)

for key in owsa_keys:
    for n in ["low", "high"]:
        insert_scenario(
            scenarios=scenarios,
            pair=define_scenario_extension(
                scenarios=bayesian_scenario,
                extensions={
                    f"{key}_{n}": {
                        key: Constant(value=tornado_ranges[key][n]),
                    }
                },
            ),
        )


# -----------------------------------------------------------
# Thesis base case uses the triangular LTB fraction of ABR. Add a paired
# structural scenario using treatment-specific absolute annual incidence
# from Toure 2022 and Zwagemaker 2021.
# -----------------------------------------------------------
# Point estimate 0.0074/PY: pooled ICH incidence for people <25 y
# (Zwagemaker et al., Blood 2021; 95% CI 0.0049-0.0111).
ltb_rate_absolute = 0.0074
absolute_ltb_scenarios = define_scenario_extension(
    scenarios=bayesian_scenario,
    extensions={
        "ltb_absolute": {
            "life_threatening_bleeding_rate": Constant(value=ltb_rate_absolute),
        }
    },
)
scenarios.extend(absolute_ltb_scenarios)

# Discounting scenario
insert_scenario(
    scenarios=scenarios,
    pair=define_scenario_extension(
        scenarios=bayesian_scenario,
        extensions={
            "is_discounting": {
                "benefits_discount_rate": Constant(value=context.simulation.discounting.utility_rate_annual),
                "costs_discount_rate": Constant(value=context.simulation.discounting.cost_rate_annual),
            },
        },
    ),
)

In [ ]:
chains = []
states = [state.value for state in HealthStates]

# NOTE:
# Identity chain uses to introduce the states and matrix shape for further runtime calculation of transition matrix per individual patient.
# This matrix will be updated on patients growth adjusting with new transitions probabilities as the patients natural mortality rate changes over time.

identity_chain = Chain(
    name="main",
    states=states,
    matrix=np.eye(
        N=len(states), M=len(states), dtype=np.float64
    ),  # Identity matrix (Cubic)
)
print(f"\n{identity_chain.matrix} \n Identity matrix")
chains.append(identity_chain)

In [ ]:
from utils import stable_hash

bundles: list[ScenarioBundle[ModelInput]] = []

for scenario in scenarios:
    scenario_seed = stable_hash(seed, scenario.name)
    scenario_params = scenario.apply_overrides(owsa_parameters)

    sampler = PSASampler(scenario_params, seed=scenario_seed)
    raw_samples = sampler.sample(sample_size)
    resolved_samples = ParameterResolver.resolve_samples(
        raw_samples, ltb_mode=ltb_mode_for_scenario(scenario.name)
    )

    inputs = [
        ParameterResolver.build_single(resolved_samples, i) for i in range(sample_size)
    ]

    bundles.append(
        ScenarioBundle(
            scenario=scenario,
            inputs=inputs,
        )
    )

    logger.info(
        "Generated %d model inputs, bundled",
        len(inputs),
        extra={"scenario": scenario.name},
    )

In [ ]:
from app.domain.worker import worker_function_batch
from app.notebook.scenario_runner import run_scenarios_in_batches

# run and write parquet files per pair
run_scenarios_in_batches(
    bundles=bundles,
    context=context,
    identity_chain=identity_chain,
    worker_function=worker_function,
    batch_size=4,
    output_dir=root / "app/cache" / "owsa" / "parquet",
    temp_dir=root / "app/cache" / "owsa" / "parquet_temp",
    options={
        "use_cached_temp": False,
    },
    engine="batch",
    batch_worker_function=worker_function_batch,
)

In [ ]:
import polars as pl

df = pl.read_parquet(
    root / "app/cache" / "owsa" / "parquet" / "all_results_combined.parquet"
)
# Each scenario has `sample_size` rows, so divide total rows by sample_size to get scenario count
scenario_count = int(df.height / sample_size)
# Each pair has 2 scenarios (on-demand and prophylaxis), so divide by 2 to get pair count
scenario_pair_count = scenario_count // 2
# Each parameter scenario has 4 variations (base, low, high for both on-demand and prophylaxis), so divide by 4 to get unique parameter scenario count
scenario_param_count = scenario_count // 4
logger.info(
    "Total scenarios: %d, Total pairs: %d, Unique parameter scenarios: %d",
    scenario_count,
    scenario_pair_count,
    scenario_param_count,
)